In [17]:
import numpy as np

y = df["middle_salary"].to_numpy(float)
sigma = y.std(ddof=1)
rho3  = np.mean(np.abs(y - y.mean())**3)

C = 0.4748  # best-known Berry–Esseen constant
for eps in (0.10, 0.05, 0.02):        # choose your own tolerances
    n_needed = ((C * rho3) / (sigma**3 * eps))**2
    print(f"ε = {eps:4.2f}  →  n ≥ {np.ceil(n_needed):.0f}")

ε = 0.10  →  n ≥ 38
ε = 0.05  →  n ≥ 151
ε = 0.02  →  n ≥ 943


In [13]:
from scipy.stats import normaltest      # add this import once, at the top

# ── Residual normality check (D'Agostino–Pearson) ───────────
k2, p = normaltest(model.resid)
print(f"D’Agostino–Pearson normality: K² = {k2:.3f},  p = {p:.4g}")

D’Agostino–Pearson normality: K² = 410.289,  p = 8.069e-90


In [35]:
import pandas as pd, numpy as np

df = (pd.read_csv("salaries_3000.csv",
                  usecols=["occupation", "gender", "age", "middle_salary"])
        .dropna())

df["occupation"] = df["occupation"].astype("category")
df["gender"]     = df["gender"].astype("category")
df["age"]        = pd.to_numeric(df["age"], errors="coerce")

# Log-transform salaries to tame positive skew (optional but recommended)
df["log_salary"] = np.log(df["middle_salary"])

In [39]:
#!/usr/bin/env python3
# deepseek_salary_audit.py  -----------------------------------------
# End-to-end bias audit of DeepSeek salary predictions
# -------------------------------------------------------------------
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.outliers_influence import OLSInfluence
from scipy.stats import shapiro, levene

# ────────────────────────── CLI ─────────────────────────────────────
parser = argparse.ArgumentParser(
    description="Audit DeepSeek salary predictions for gender-occupation bias"
)
parser.add_argument(
    "--csv", type=Path, required=True,
    help="/Users/vitusthaulow/Documents/GitHub/Project-in-Stat.-Eval.-02445/salaries_3000.csv"
)
args = parser.parse_args()

# ────────────────────── 1. Load + tidy  ────────────────────────────
cols = ["occupation", "gender", "age", "middle_salary"]
df   = pd.read_csv(args.csv, usecols=cols).dropna()

df["occupation"] = df["occupation"].astype("category")
df["gender"]     = df["gender"].astype("category")
df["age"]        = pd.to_numeric(df["age"], errors="coerce")

df["log_salary"] = np.log(df["middle_salary"])

print(f"Loaded {len(df):,} rows   •   "
      f"{df['occupation'].nunique()} occupations   •   "
      f"{df['gender'].unique()} genders   •   age {df['age'].min()}–{df['age'].max()}")

# ───────────────────── 2. Fit full model  ──────────────────────────
formula_full = "log_salary ~ C(occupation)*C(gender)*age"
model_full   = smf.ols(formula_full, data=df).fit(cov_type="HC3")

# ANCOVA / ANOVA table (Type II)
aov = anova_lm(model_full, typ=2)
aov["partial_eta2"] = aov["sum_sq"] / (aov["sum_sq"] + aov.loc["Residual", "sum_sq"])

print("\n=== Robust full model (HC3) summary ===")
print(model_full.summary().tables[0])           # basic header
print(model_full.summary().tables[1][:10])      # first few coefficients

print("\n=== Type-II ANOVA (robust model fit) ===")
print(aov[["F", "PR(>F)", "partial_eta2"]].round(3))

# ───────────────── 3. Diagnostics  ────────────────────────────────
stud_resid   = OLSInfluence(model_full).resid_studentized_internal
w, p_shapiro = shapiro(stud_resid)
print(f"\nShapiro–Wilk on studentised residuals: W = {w:.3f}, p = {p_shapiro:.3g}")

group_vals = [
    g["log_salary"].values
    for _, g in df.groupby(["occupation", "gender"], observed=True)
]
W_levene, p_levene = levene(*group_vals, center="median")
print(f"Brown–Forsythe (median-centered Levene): W = {W_levene:.2f}, p = {p_levene:.3g}")

# ──────────────── 4. Plots (saved to PNGs) ─────────────────────────
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 150

# (a) Bar-plot of marginal means (raw salary)
mean_tbl = (
    df.groupby(["occupation", "gender"], observed=True)["middle_salary"]
      .mean()
      .unstack()
)
mean_tbl.plot(kind="bar", rot=0, figsize=(8, 5),
              color=["#1f77b4", "#ff7f0e"])
plt.ylabel("Predicted salary (DKK)")
plt.title("Average DeepSeek salary by occupation and gender")
plt.tight_layout()
plt.savefig("salary_barplot.png", dpi=300)
plt.close()

# (b) Q–Q plot
import statsmodels.api as sm
sm.qqplot(stud_resid, line="45", fit=True)
plt.title("Q–Q plot of studentised residuals")
plt.tight_layout()
plt.savefig("qq_plot.png", dpi=300)
plt.close()

# (c) Histogram of residuals
plt.hist(stud_resid, bins="auto", edgecolor="k")
plt.xlabel("Studentised residual")
plt.ylabel("Frequency")
plt.title("Histogram of residuals")
plt.tight_layout()
plt.savefig("resid_hist.png", dpi=300)
plt.close()

print("\nPlots saved: salary_barplot.png, qq_plot.png, resid_hist.png")

# ───────────── 5. Simple-effects helper (preview) ──────────────────
# Example: gender gap within each occupation at age=40
from statsmodels.stats.anova import simple_effects
simple = simple_effects(model_full, ["gender"], at={"age": 40})
print("\n=== Simple gender effects at Age = 40 ===")
print(simple)

print("\nDONE – see console output and PNG files.")


usage: ipykernel_launcher.py [-h] --csv CSV
ipykernel_launcher.py: error: the following arguments are required: --csv


SystemExit: 2